In [2]:
import spacy
import pandas as pd
from spacy.tokens import DocBin
from sklearn.model_selection import train_test_split
import ast

In [3]:
bio = pd.read_csv("../data/voicepay_bio_dataset.csv")
bio.head()

,text,amount,currency,recipient,bill_type,intent,tokens,ner_tags
0,سدد فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['سدد', 'فاتورة', 'الكهرباء']","['O', 'O', 'B-BILL_TYPE']"
1,قم بدفع فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['قم', 'بدفع', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
2,ادفع قيمة فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['ادفع', 'قيمة', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
3,قم بسداد فاتورة الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['قم', 'بسداد', 'فاتورة', 'الكهرباء']","['O', 'O', 'O', 'B-BILL_TYPE']"
4,سدد مستحقات الكهرباء,NaN,NaN,NaN,الكهرباء,pay_bill,"['سدد', 'مستحقات', 'الكهرباء']","['O', 'O', 'B-BILL_TYPE']"


In [5]:
# convert the BIO tags to spacy format
def bio_to_spacy_format(df):
    """
    Converts a DataFrame with tokens and BIO tags to spaCy training format.
    Returns: list of tuples (text, {"entities": [(start, end, label), ...]})
    """
    TRAIN_DATA = []

    for _, row in df.iterrows():
        text = row["text"]

        # Convert string representation of list to real list (if needed)
        tokens = row["tokens"]
        ner_tags = row["ner_tags"]

        if isinstance(tokens, str):
            tokens = ast.literal_eval(tokens)
        if isinstance(ner_tags, str):
            ner_tags = ast.literal_eval(ner_tags)

        entities = []
        current_entity = None
        char_index = 0  # keep track of character position in text

        # Loop over tokens and BIO tags
        for token, tag in zip(tokens, ner_tags):
            start = text.find(token, char_index)  # start char index
            end = start + len(token)              # end char index
            char_index = end

            if tag.startswith("B-"):
                # Save previous entity if exists
                if current_entity:
                    entities.append(current_entity)

                # Start new entity
                label = tag[2:]
                current_entity = [start, end, label]

            elif tag.startswith("I-") and current_entity:
                # Extend current entity
                current_entity[1] = end

            else:
                # O tag or new entity
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        # Append last entity if exists
        if current_entity:
            entities.append(current_entity)

        TRAIN_DATA.append((text, {"entities": entities}))

    return TRAIN_DATA

In [6]:
TRAIN_DATA = bio_to_spacy_format(bio)
print("Example converted data:")
print(TRAIN_DATA[380])

Example converted data:
('حوّل مبلغ 10 دنانير لحمزة', {'entities': [[10, 12, 'AMOUNT'], [13, 19, 'CURRENCY'], [20, 25, 'RECIPIENT']]})


In [ ]:
# dev set (development) is used for evaluation during training -> Dev set = Validation set
train_data, dev_data = train_test_split(TRAIN_DATA, test_size=0.2, random_state=42)

In [ ]:
def save_docbin(data, filename, nlp):
    """
    Saves a list of spaCy training examples into a .spacy file.
    """
    doc_bin = DocBin()
    for text, annotations in data:
        doc = nlp.make_doc(text)
        ents = []
        for start, end, label in annotations["entities"]:
            span = doc.char_span(start, end, label=label)
            if span is not None:
                ents.append(span)
        doc.ents = ents
        doc_bin.add(doc)
    doc_bin.to_disk(filename)
    print(f"Saved {filename} with {len(data)} examples")


In [3]:
nlp = spacy.blank("ar")

In [ ]:
save_docbin(train_data, "train.spacy", nlp)
save_docbin(dev_data, "dev.spacy", nlp)

In [4]:
nlp = spacy.load("output/model-best")
"""
============================= Training pipeline =============================
Pipeline: ['tok2vec', 'ner']
Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     79.61    0.00    0.00    0.00    0.00
 14     200         73.25   2306.85   89.05   88.41   89.71    0.89
 33     400        110.09    340.42   88.32   87.68   88.97    0.88
 55     600        133.90    171.48   88.00   87.05   88.97    0.88
 81     800        142.99    115.59   88.24   88.24   88.24    0.88
114    1000        156.18    115.35   88.64   88.32   88.97    0.89
153    1200        123.78    132.48   87.91   87.59   88.24    0.88
202    1400        108.33    131.54   88.24   88.24   88.24    0.88
260    1600        119.46    145.90   88.24   88.24   88.24    0.88
326    1800        179.31    174.01   86.55   85.61   87.50    0.87
✔ Saved pipeline to output directory
output/model-last

"""

"\n============================= Training pipeline =============================\nPipeline: ['tok2vec', 'ner']\nInitial learn rate: 0.001\nE    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE \n---  ------  ------------  --------  ------  ------  ------  ------\n  0       0          0.00     79.61    0.00    0.00    0.00    0.00\n 14     200         73.25   2306.85   89.05   88.41   89.71    0.89\n 33     400        110.09    340.42   88.32   87.68   88.97    0.88\n 55     600        133.90    171.48   88.00   87.05   88.97    0.88\n 81     800        142.99    115.59   88.24   88.24   88.24    0.88\n114    1000        156.18    115.35   88.64   88.32   88.97    0.89\n153    1200        123.78    132.48   87.91   87.59   88.24    0.88\n202    1400        108.33    131.54   88.24   88.24   88.24    0.88\n260    1600        119.46    145.90   88.24   88.24   88.24    0.88\n326    1800        179.31    174.01   86.55   85.61   87.50    0.87\n✔ Saved pipeline to output direct

In [ ]:
samples_p2p = [
    "حوّل 10 دنانير لأحمد",
    "حوّل 25 دينار لريم",
    "حوّل مبلغ 50 دينار لمراد",
    "حوّل 8 دنانير لسمير",
    "حوّل 13 دينار لفرح"
]

samples_pay_bill = [
    "سدد فاتورة الكهرباء",
    "ادفع قيمة فاتورة المياه",
    "قم بدفع فاتورة الإنترنت",
    "سدد مستحقات الغاز",
    "ادفع اشتراك الهاتف لهذا الشهر"
]

samples_mixed = [
    "حوّل 100 دينار لسمير وادفع فاتورة الكهرباء",
    "قم بسداد 15 دينار لفرح وادفع فاتورة الإنترنت",
    "ادفع 50 دنانير فاتورة الهاتف",
    "حوّل مبلغ 20 دينار لريم",
    "سدد فاتورة المياه المتأخرة" 
]

samples = samples_p2p + samples_pay_bill + samples_mixed + ["مبلغ 10 دنانير حمزة"]
for text in samples:
    doc = nlp(text)
    print(f"\nText: {text}")
    for ent in doc.ents:
        print(ent.text, ent.label_)


Text: حوّل 10 دنانير لأحمد
10 AMOUNT
دنانير CURRENCY
لأحمد RECIPIENT

Text: حوّل 25 دينار لريم
25 AMOUNT
دينار CURRENCY
لريم RECIPIENT

Text: حوّل مبلغ 50 دينار لمراد
50 AMOUNT
دينار CURRENCY
لمراد RECIPIENT

Text: حوّل 8 دنانير لسمير
8 AMOUNT
دنانير CURRENCY
لسمير RECIPIENT

Text: حوّل 13 دينار لفرح
13 AMOUNT
دينار CURRENCY
لفرح RECIPIENT

Text: سدد فاتورة الكهرباء
الكهرباء BILL_TYPE

Text: ادفع قيمة فاتورة المياه
المياه BILL_TYPE

Text: قم بدفع فاتورة الإنترنت
الإنترنت BILL_TYPE

Text: سدد مستحقات الغاز
الغاز BILL_TYPE

Text: ادفع اشتراك الهاتف لهذا الشهر
اشتراك الهاتف BILL_TYPE

Text: حوّل 100 دينار لسمير وادفع فاتورة الكهرباء
100 AMOUNT
دينار CURRENCY
لسمير RECIPIENT
الكهرباء BILL_TYPE

Text: قم بسداد 15 دينار لفرح وادفع فاتورة الإنترنتقم بسداد 15 دينار لفرح وادفع فاتورة الإنترنت
15 AMOUNT
دينار CURRENCY
لفرح RECIPIENT
فاتورة الإنترنتقم BILL_TYPE
15 AMOUNT
دينار CURRENCY
لفرح RECIPIENT
الإنترنت BILL_TYPE

Text: ادفع 50 دنانير فاتورة الهاتف
50 AMOUNT
دنانير CURRENCY
الهاتف BILL_TYP